## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [7]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [8]:
load_dotenv(override=True)

True

In [9]:
print(bool(os.getenv("OPENROUTER_API_KEY")))

True


In [10]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-inf
Groq API Key exists and begins gsk_


In [11]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [12]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [13]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

STEP 3: Create a model object

In [14]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [15]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [16]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [17]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [18]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [19]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [20]:
tools = [tool1, tool2, tool3, send_email_tool]

In [21]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [22]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

Sent the best email.


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [23]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [24]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [25]:
email = """
Hi [first_name],

I'm ghanta hitting you up to see if you'd like to buy our product.
It's really great.
You'll miss out if you don't buy it.

Laters.

Ed
"""

In [26]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [27]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [28]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [29]:
@output_guardrail
async def check_email(
    ctx,
    agent,
    output
):
    guardrail_result = await Runner.run(
        checker,
        output
    )

    check = guardrail_result.final_output

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.contains_prohibited_content
    )

In [31]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[check_email])

In [32]:
from agents import OutputGuardrailTripwireTriggered

try:
    result = await Runner.run(
        sales_agent_cowboy,
        "Write a cold sales email"
    )

    print(result.final_output)

except OutputGuardrailTripwireTriggered as e:
    print("Tripwire triggered")

    print(e.guardrail_result)

AttributeError: 'EmailReview' object has no attribute 'contains_prohibited_content'

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [33]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


Howdy partner. If you’re lookin’ to wrangle some results and stop your team from gettin’ bucked off by bad data, you’ve come to the right place. At ComplAI, we don't just clear the brush—we pave the whole trail for your compliance needs.

Here is the cold email I’ve been usin’ to round up the herd. It’s short, sharp, and cuts right to the bone.

***

**Subject:** A quicker way to clear the compliance fence?

Howdy [Prospect Name],

Most folks I talk to in [Prospect's Industry] spend more time wrestling with compliance paperwork than they do actually runnin’ their business. It’s like tryin’ to herd cats in a thunderstorm.

I’m reaching out because ComplAI helps teams like yours automate the grit-work so you can focus on the horizon. We’ve helped folks cut their manual review time down by [X]%, and I reckon we could do the same for you.

You got five minutes later this week to chat about how we can lighten your load?

Best,

[Your Name]
Sales Manager, ComplAI

***

**Why this works, part

In [34]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [35]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [36]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [37]:
CODE_DIR

PosixPath('/Users/admin/Downloads/agent/agents/2_openai/code')

In [38]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [39]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [40]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [41]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [42]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

Fixed the bug in `orders.py`.

- Replaced `dict.fromkeys(..., [])`, which caused all customers to share the same list.
- Switched to `setdefault` so each `customer_id` gets its own order list.
- Wrote the corrected file to `/Users/admin/Downloads/agent/agents/2_openai/output/orders.py`.

If you want, I can also add a quick test for the grouping behavior.


## OPTIONAL EXTRA: MCP Teaser!

In [47]:
from agents.mcp import MCPServerStreamableHttp


In [44]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [45]:
agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

As of my last update, I don't have specific information about features or updates in the OpenAI Agents SDK, including the SandboxAgents feature or the role of the Manifest object as of May 2026. For the most accurate and current information, I recommend checking the official OpenAI documentation or announcements related to this feature.


In [51]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    print(server)
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

In the new **SandboxAgents feature** of the **OpenAI Agents SDK**, the **Manifest object** plays a crucial role in defining the environment and permissions for a SandboxAgent. Specifically, it allows you to:

1. **Declare Entries**: The Manifest can specify directories, such as a local directory for the agent to access, thus enabling it to read from and write to specific paths within the sandboxed environment.

2. **Control Permissions**: You can grant different permissions to users for various directories. This controls access levels (read, write, execute) for each entry within the sandbox, ensuring security and compliance with user identity settings.

3. **Set User Context**: The Manifest can define user identities that the SandboxAgent operates under, allowing it to enforce user-based permission boundaries during execution.

4. **Grant Extra Path Access**: It enables the configuration of additional absolute path grants for access beyond the workspace, which can be read-only if neede

And see the traces:

https://platform.openai.com/traces
